In [51]:
# !cp -r /kaggle/input/deeplabv3 .
# !pip install -U albumentations

In [52]:
import torch
import torch.nn as nn
import os

from deeplabv3.datasets import get_images, get_dataset, get_data_loaders
from deeplabv3.engine import train, validate
from deeplabv3.config import ALL_CLASSES, LABEL_COLORS_LIST
from deeplabv3.utils import save_model, SaveBestModel, save_plots, SaveBestModelIOU
from torch.optim.lr_scheduler import StepLR

In [53]:
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

In [54]:
from torchvision.models.segmentation import deeplabv3_resnet50

# Load model
model = deeplabv3_resnet50(weights='DEFAULT')
model.classifier[4] = torch.nn.Conv2d(256, 2, 1)  
model.aux_classifier[4] = torch.nn.Conv2d(256, 2, 1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [55]:
out_dir = os.path.join('.', 'outputs')
out_dir_valid_preds = os.path.join('.', 'outputs', 'valid_preds')
os.makedirs(out_dir, exist_ok=True)
os.makedirs(out_dir_valid_preds, exist_ok=True)

In [56]:
# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

41,994,308 total parameters.
41,994,308 training parameters.


In [57]:
train_images, train_masks, valid_images, valid_masks = get_images(
    root_path = 'C:/PythonProjects/plant_disease_class/leaf_segmentation/leaf_disease_segmentation/orig_data'
)

classes_to_train = ALL_CLASSES

train_dataset, valid_dataset = get_dataset(
    train_images, 
    train_masks,
    valid_images,
    valid_masks,
    ALL_CLASSES,
    classes_to_train,
    LABEL_COLORS_LIST,
    img_size=512
)

train_dataloader, valid_dataloader = get_data_loaders(
    train_dataset, valid_dataset, batch_size=8
)

# Initialize `SaveBestModel` class.
save_best_model = SaveBestModel()
save_best_iou = SaveBestModelIOU()
# LR Scheduler.
scheduler = StepLR(optimizer, step_size=20, gamma=0.1)

In [58]:
ckpt = "C:/PythonProjects/plant_disease_class/checkpoints/model_checkpoint.pth"
if os.path.exists(ckpt):
        ckpt = torch.load(ckpt)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch']
else:
    epoch_start = 0

In [59]:
num_epochs = 60
train_loss, train_pix_acc, train_miou = [], [], []
valid_loss, valid_pix_acc, valid_miou = [], [], []

for epoch in range (epoch_start, num_epochs):
    print(f"Epoch: {epoch + 1}")
    train_epoch_loss, train_epoch_pixacc, train_epoch_miou = train(
        model,
        train_dataset,
        train_dataloader,
        device,
        optimizer,
        criterion,
        classes_to_train
    )
    valid_epoch_loss, valid_epoch_pixacc, valid_epoch_miou = validate(
        model,
        valid_dataset,
        valid_dataloader,
        device,
        criterion,
        classes_to_train,
        LABEL_COLORS_LIST,
        epoch,
        ALL_CLASSES,
        save_dir=out_dir_valid_preds
    )
    train_loss.append(train_epoch_loss)
    train_pix_acc.append(train_epoch_pixacc)
    train_miou.append(train_epoch_miou)
    valid_loss.append(valid_epoch_loss)
    valid_pix_acc.append(valid_epoch_pixacc)
    valid_miou.append(valid_epoch_miou)

    save_best_model(
        valid_epoch_loss, epoch, model, out_dir, name='model_loss'
    )
    save_best_iou(
        valid_epoch_miou, epoch, model, out_dir, name='model_iou'
    )

    print(
        f"Train Epoch Loss: {train_epoch_loss:.4f},",
        f"Train Epoch PixAcc: {train_epoch_pixacc:.4f},",
        f"Train Epoch mIOU: {train_epoch_miou:4f}"
    )
    print(
        f"Valid Epoch Loss: {valid_epoch_loss:.4f},", 
        f"Valid Epoch PixAcc: {valid_epoch_pixacc:.4f}",
        f"Valid Epoch mIOU: {valid_epoch_miou:4f}"
    )
    scheduler.step()
    print('-' * 50)

save_model(num_epochs, model, optimizer, criterion, out_dir, name='model')
# Save the loss and accuracy plots.
save_plots(
    train_pix_acc, valid_pix_acc, 
    train_loss, valid_loss,
    train_miou, valid_miou, 
    out_dir
)
print('Training finished.')

Epoch: 1
Training


 24%|████▊               | 15/62 [00:35<01:50,  2.35s/it]


KeyboardInterrupt: 

In [ ]:
from deeplabv3.utils import get_segment_labels, draw_segmentation_map, image_overlay
from PIL import Image
import cv2
import numpy as np
from deeplabv3.config import ALL_CLASSES
from deeplabv3.model import prepare_model

In [ ]:
out_dir = os.path.join('.', 'outputs', 'inference_results')
os.makedirs(out_dir, exist_ok=True)

# Set computation device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = prepare_model(num_classes=len(ALL_CLASSES)).to(device)
ckpt = torch.load('outputs/model.pth')
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(device)

image_path = "C:/PythonProjects/plant_disease_class/leaf_segmentation/leaf_disease_segmentation/orig_data/valid_images"
images = os.listdir(image_path)
image_name = "00000.jpg"
image = Image.open(os.path.join(image_path, image_name))

image = image.resize((512, 512))

# Do forward pass and get the output dictionary.
outputs = get_segment_labels(image, model, device)
outputs = outputs['out']
segmented_image = draw_segmentation_map(outputs)
final_image = cv2.cvtColor(image_overlay(image, segmented_image), cv2.COLOR_BGR2RGB)

mask_path = "C:/PythonProjects/plant_disease_class/leaf_segmentation/leaf_disease_segmentation/orig_data/valid_masks"
mask_image = Image.open(os.path.join(mask_path, image_name[:-4]+".png")).convert("RGB")
mask_image = mask_image.resize((512, 512))
mask_image = np.array(mask_image)

true_image = cv2.cvtColor(image_overlay(image, mask_image), cv2.COLOR_BGR2RGB)

In [ ]:
import matplotlib.pyplot as plt

# Display the true mask and predicted mask side-by-side
plt.figure(figsize=(12, 6))

# Plot the predicted mask
plt.subplot(2, 2, 1)
plt.title('Predicted Mask')
plt.imshow(segmented_image)
plt.axis('off')

# Plot the true mask
plt.subplot(2, 2, 2)
plt.title('True Mask')
plt.imshow(mask_image)
plt.axis('off')

# Plot the predicted image
plt.subplot(2, 2, 3)
plt.imshow(final_image)
plt.axis('off')

# Plot the true image
plt.subplot(2, 2, 4)
plt.imshow(true_image)
plt.axis('off')

plt.show()